[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_strain.ipynb)

# Two-Phase Composite RVE — Linear-Elastic Strain Solve

A minimal walkthrough of FFTjax's strain-based Newton-CG elastic solver
(`solvers.mechanical.strain_nw_cg.solve_elastic`) on a **two-phase composite**: a glass-fiber
reinforcement in an epoxy matrix, arranged in a square-packed pattern via
`generation.rve.make_square_composite_rve`, under a prescribed macroscopic strain.

Because the two phases have a large stiffness contrast (~23x), the reference-medium correction is
nontrivial — the Newton-CG solve actually iterates, redistributing stress between the stiff fibers
and the compliant matrix.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    print("Running locally — using the local src/ checkout.")

In [ ]:
import sys
sys.path.insert(0, "../src")
import os

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np

from generation.rve import make_square_composite_rve
from operators.green import build_freq_grid, build_green_operator
from mat_models.elastic import LinearElasticIsotropic, assemble_C_field
from solvers.mechanical.strain_nw_cg import solve_elastic
from post.fields import field_to_grid, von_mises, compute_displacement
from post.io import IncrementalWriter, to_voigt

import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

`generation.rve.make_square_composite_rve` builds a square-packed 2-fiber RVE: a matrix phase with
circular fiber cross-sections arranged on a square lattice, extruded along Z into a 3-D voxel grid. Here we use a 10-voxel-thick RVE, with a fiber volume fraction of 0.5, fiber radius of 5 μm, and voxel spacing of 0.5 μm.
Setting `nz=1` reduces the geometry to a 2-D-like slab (uniformly extruded along Z); combined with a macroscopic strain that has no out-of-plane (Z) components -- as prescribed below -- this gives a plane-strain solve.

In [ ]:
phase_np, N, n, L, phi_act = make_square_composite_rve(
    phi=0.5, r_fiber=0.005, spacing=0.0002, N_min=32, nz=10,
)
Nv = int(np.prod(n))

print("grid n :", n)
print("domain L [mm]:", tuple(float(Li) for Li in L))
print("fiber volume fraction (actual):", phi_act)

In [ ]:
# Vizualize the fibre cross-section in the XY plane (Z=0)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r")
ax.set_title(f"Fiber cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [voxel]")
ax.set_ylabel("y [voxel]")
plt.show()

## Materials and stiffness field

A glass fiber in an epoxy matrix — a common, high-contrast (~23x stiffness ratio) composite.

In [ ]:
matrix = LinearElasticIsotropic(E=3.0e3,  nu=0.35, name="epoxy matrix")
fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fiber")

phase = jnp.array(phase_np.reshape(-1))   # 0 = matrix, 1 = fibre
C_field = assemble_C_field([matrix, fiber], phase)

print(matrix)
print(fiber)

## Frequency grid, Green's operator, and solve

The reference medium is the average of the two phases' Lamé parameters — a reasonable choice when
neither phase dominates. We use Willot's `rotated` discretisation scheme for the Green's operator,
which reduces spurious oscillations at phase interfaces compared to the `standard` scheme.

In [ ]:
L_mm = tuple(float(Li) for Li in L)
dx = tuple(Li / ni for Li, ni in zip(L_mm, n))
xi_flat = build_freq_grid(n, L_mm)

lam0 = 0.5 * (matrix.lam + fiber.lam)
mu0  = 0.5 * (matrix.mu  + fiber.mu)
G_glob = build_green_operator(xi_flat, lam0, mu0, scheme="rotated", dx=dx)

# we apply a small shear strain in the XY plane, with zero normal strains
eps_bar = jnp.array([
    [0.0, 1.0e-3, 0.0],
    [1.0e-3, 0.0, 0.0],
    [0.0,    0.0, 0.0],
])

eps, sigma, delta, it, converged = solve_elastic(
    n, C_field, G_glob, eps_bar, toler_lin=1e-6, maxiter=1000,
)

print("CG iterations:", int(it))
print("converged     :", bool(converged))
print("tau_xy (avg) :", float(jnp.mean(sigma[1, 0])), "MPa")

The Newton-CG solve takes real iterations to converge — the correction field is doing real work redistributing stress between the stiff fibres and the compliant matrix.

## Post-processing

Now we can visualize the results and also export them as a `.xdmf`/`.h5` pair for further
post-processing in ParaView or other visualization software, via FFTjax's `IncrementalWriter`
(the project-wide standard for field-data output). Every field here -- displacement, strain,
stress, phase -- is evaluated on the same voxel grid, so all of them are written voxel-centered
(`Center="Cell"`); there's no FEM-style node/cell split in this spectral scheme, so there's nothing
to gain from writing displacement at a different resolution than everything else.

In [ ]:
# Post-processing
# return the fields to a 3-D grid for visualization and export
eps_grid   = field_to_grid(eps, n)
sigma_grid = field_to_grid(sigma, n)
u_grid     = compute_displacement(eps, eps_bar, xi_flat, n, dx)
vm_grid    = von_mises(sigma_grid)

eps_voigt   = to_voigt(eps_grid).astype(np.float64)
sigma_voigt = to_voigt(sigma_grid).astype(np.float64)

display the stress and strain field in matplotlib.

In [ ]:
VOIGT_LABELS = ["x", "y", "z", "xy", "xz", "yz"]
extent = [0.0, n[0] * dx[0], 0.0, n[1] * dx[1]]  # physical [mm] extent, binned by voxel size dx

fig, axes = plt.subplots(3, 3, figsize=(10, 9))

im = axes[0, 0].imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r", extent=extent)
axes[0, 0].set_title("Fiber phase")
fig.colorbar(im, ax=axes[0, 0])

im = axes[0, 1].imshow(u_grid[:, :, 0, 0].T, origin="lower", cmap="plasma", extent=extent)
axes[0, 1].set_title(r"Displacement $u_x$ [mm]")
fig.colorbar(im, ax=axes[0, 1], format="%.1e")

im = axes[0, 2].imshow(u_grid[:, :, 0, 1].T, origin="lower", cmap="plasma", extent=extent)
axes[0, 2].set_title(r"Displacement $u_y$ [mm]")
fig.colorbar(im, ax=axes[0, 2], format="%.1e")

for idx, i in enumerate([0, 1, 3]):
    eps_plot = eps_voigt[:, :, 0, i]
    label = rf"$\varepsilon_{{{VOIGT_LABELS[i]}}}$"
    if i == 3:  # shear component: report engineering shear strain gamma = 2*epsilon
        label = rf"$\gamma_{{{VOIGT_LABELS[i]}}}$"

    im = axes[1, idx].imshow(eps_plot.T, origin="lower", cmap="plasma", extent=extent)
    axes[1, idx].set_title(f"Strain {label}")
    fig.colorbar(im, ax=axes[1, idx], format="%.1e")

    label = rf"$\sigma_{{{VOIGT_LABELS[i]}}}$"
    if i == 3:
        label = rf"$\tau_{{{VOIGT_LABELS[i]}}}$"
    im = axes[2, idx].imshow(sigma_voigt[:, :, 0, i].T, origin="lower", cmap="plasma", extent=extent)
    axes[2, idx].set_title(f"Stress {label}")
    fig.colorbar(im, ax=axes[2, idx], format="%.1f")

for ax in axes.flat:
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")

plt.tight_layout()
plt.show()

In [ ]:
output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)

with IncrementalWriter(f"{output_dir}/composite_rve", grid_shape=n, grid_spacing=dx) as w:
    w.write_increment(0, {
        "phase":        phase_np.astype(np.float64),
        "displacement": u_grid.astype(np.float64),
        "strain":       to_voigt(eps_grid).astype(np.float64),
        "stress":       to_voigt(sigma_grid).astype(np.float64),
        "von_mises":    vm_grid.astype(np.float64),
    }, time=0.0)

print(f"Wrote {output_dir}/composite_rve.h5")
print(f"      {output_dir}/composite_rve.xdmf")
print("Open the .xdmf in ParaView with the 'Xdmf3ReaderT' reader.")

## Next steps

- Sweep grid size for the composite RVE above — the [Benchmark](https://choROPeNt.github.io/FFTjax/documentation/benchmark#linear-elastic-strain-solve) page does exactly this and times it.
- See [`lin-elastic_mixed-BC.ipynb`](./lin-elastic_mixed-BC.ipynb) for the same geometry/materials
  under a free-lateral-surface uniaxial-*stress* condition instead — a more realistic mechanical
  test, and a heterogeneous case where the strain-based mixed-BC solver
  (`solvers.mechanical.strain_nw_cg.dstrain_nw_cg_mixed`) isn't valid, so the displacement-based
  `ddisp_nw_cg` is used instead.